In [1]:
import sys
from pathlib import Path

CWD = Path(__name__).resolve().parent
sys.path.append(CWD)

DATASET_FILE = CWD / "result.json"
# OUT_CHECKPOINT_FILE = CWD / "leanrag_checkpoint.json"
# USE_CHECKPOINT_AS_CACHE = True # prefer data in the checkpoint file over re-computing?

secrets = CWD / "secrets.env"

if not secrets.is_file():
    raise ValueError(f"secrets file at '{secrets}' does not exist")

from dotenv import load_dotenv
load_dotenv(secrets)

# (START WITH G0 IN MEMGRAPH) ...

True

In [2]:
#  - - TEMP - -
# from utils import mg_driver
# import importlib
# importlib.reload(mg_driver)

# await mg_driver.init()

# x = await mg_driver.get_ancestor_chain("proposed_architecture")

# print(x)

In [3]:
EMBED_MODEL = "bedrock/amazon.titan-embed-text-v2:0"
G0_EMBEDDINGS_FILE = CWD / "g0_embeddings.json"

USER_PROMPT = "What architectural modification did the authors make to the GRU model from BotScreen?"

In [4]:
# load g0 entity embeddings from file
import json
from utils.models import EntityDescEmbed
entity_embeddings:list[EntityDescEmbed] = []
with open(G0_EMBEDDINGS_FILE, "r") as ef:
    entity_embeddings = json.load(ef)

print(len(entity_embeddings))
if len(entity_embeddings) > 0:
    print(entity_embeddings[0])

620
{'key': 'notable_improvement_over_a_gated_recurrent_network_on_the_same_data', 'desc_embed': [-0.03835173323750496, 0.058729853481054306, 0.017201755195856094, 0.03337221220135689, 0.02733534388244152, 0.03618475794792175, 0.031965211033821106, -0.021189281716942787, 0.014627652242779732, 0.002278724918141961, 0.034298013895750046, 0.0031000208109617233, -0.0070237028412520885, 0.031445588916540146, 0.003352146130055189, -0.006819467525929213, -0.034976791590452194, 0.04537922888994217, 0.049283985048532486, 0.03968800604343414, -0.039290476590394974, 0.01918012835085392, 0.08892661333084106, 0.03271022066473961, -0.01992572471499443, -0.010040604509413242, -0.024277418851852417, -0.021399766206741333, -0.00635359063744545, -0.027371767908334732, -0.005664876662194729, 0.06918687373399734, -0.03908133506774902, 0.03585239127278328, -0.029931705445051193, 0.07606083154678345, 0.08523454517126083, 0.030158722773194313, 0.026728564873337746, 0.041039615869522095, 0.039543382823467255,

In [5]:
import litellm

resp = await litellm.aembedding(model=EMBED_MODEL, input=USER_PROMPT)
data = resp['data'][0]
prompt_embedding:list[float] = data['embedding']
print(len(prompt_embedding))

1024


In [6]:
import numpy as np

# dense vector search function
def search_dense(query_vec: list[float], entity_store: list[dict], topk: int= 10):
    if topk <= 0:
        return []

    # Convert to NumPy arrays
    query = np.asarray(query_vec, dtype=np.float32)
    entity_matrix = np.asarray(
        [item["desc_embed"] for item in entity_store], dtype=np.float32
    )

    # Normalize to use cosine similarity (handle zero vectors defensively)
    q_norm = np.linalg.norm(query)
    if q_norm == 0:
        raise ValueError("query vector has zero norm")
    query /= q_norm

    e_norms = np.linalg.norm(entity_matrix, axis=1, keepdims=True)
    e_norms[e_norms == 0] = 1.0
    entity_matrix = entity_matrix / e_norms

    # Dot product gives cosine similarity
    sims = entity_matrix @ query

    # Grab top-k indices
    k = min(topk, len(entity_store))
    top_idx = np.argpartition(-sims, k - 1)[:k]
    top_idx = top_idx[np.argsort(-sims[top_idx])]

    return [
        {**entity_store[i], "score": float(sims[i])}
        for i in top_idx
    ]

seed_entities = search_dense(prompt_embedding, entity_embeddings)

In [7]:
[s['key'] for s in seed_entities]

['gated_recurrent_unit_gru_architecture',
 'model_in_6',
 'gru',
 'bidirectional_gru',
 'gru_based_model',
 'authors_of_botscreen',
 'bidirectional_stacked_gated_recurrent_unit',
 'the_bidirectional_gru_based_network',
 '6',
 'as_described_in_section_3_2']

In [ ]:
# assemble ancestor chains
from utils import mg_driver
await mg_driver.init()

ancestor_chains = []
for s in seed_entities:
    chain = await mg_driver.get_ancestor_chain(s['key'])
    ancestor_chains.append(chain)

[[{'desc': 'The Gated Recurrent Unit (GRU) architecture is a type of recurrent neural network (RNN) that is designed to address some of the challenges associated with traditional RNNs, such as vanishing gradients. GRUs are known for their efficiency and performance in sequence modeling tasks, and in this context, they are being tailored for the specific use case of aimbot pattern detection from movement data in multiplayer games.',
   'key': 'gated_recurrent_unit_gru_architecture',
   'layer': 0,
   'name': 'gated_recurrent_unit_gru_architecture'},
  {'desc': 'The Multiplayer Competitive Gaming Ecosystem encompasses the various elements and interactions within multiplayer games, particularly focusing on player-versus-player (PvP) combat and the detection of cheating behaviors. This ecosystem includes the participants (players), the competitive activities they engage in, the skills required for success, and the technological measures employed to ensure fair play.',
   'key': 'multiplaye